# Agent Sheep (Phase 3: MQTT Publisher)

This notebook runs the sheep-only simulation and publishes per-tick messages to MQTT topics.

Tick order stays the same: move → eat → reproduce → die → regrow grass.

It loads settings via `simulated_city.config.load_config()` and uses `MqttConnector` + `MqttPublisher` for publishing.

In [1]:
# Cell purpose: import helpers, load config, and connect to MQTT broker.
from datetime import datetime, timezone
import json
import time

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher
from simulated_city.sheep_wolf_models import SheepSimulationParams, run_simulation

config = load_config()
print(f"Loaded config. Primary MQTT profile: {config.mqtt.host}:{config.mqtt.port}")

connector = MqttConnector(config.mqtt, client_id_suffix="agent-sheep-phase3")
connector.connect()
connected = connector.wait_for_connection(timeout=5)
print(f"MQTT connected: {connected}")

publisher = MqttPublisher(connector)

base_topic = config.mqtt.base_topic.rstrip("/")
tick_topic = f"{base_topic}/sim/tick"
sheep_state_topic = f"{base_topic}/sim/sheep/state"
events_topic = f"{base_topic}/sim/events"
start_control_topic = f"{base_topic}/sim/control/commands"
start_requested = False
pending_predation_by_tick = {}


def on_agent_message(client, userdata, msg):
    global start_requested

    try:
        payload = json.loads(msg.payload.decode("utf-8"))
    except Exception:
        return

    if msg.topic == start_control_topic:
        command = str(payload.get("command", "")).strip().lower()
        if command == "start":
            start_requested = True
            print(
                f"START command received on {msg.topic} at "
                f"{datetime.now(timezone.utc).isoformat()}"
            )
        return

    if msg.topic == events_topic:
        if str(payload.get("source", "")).strip().lower() != "wolf":
            return
        event_tick = int(payload.get("tick", 0))
        estimated_predation = max(0, int(payload.get("estimated_predation", 0)))
        if event_tick <= 0 or estimated_predation <= 0:
            return
        pending_predation_by_tick[event_tick] = (
            int(pending_predation_by_tick.get(event_tick, 0)) + estimated_predation
        )
        print(
            f"Queued wolf predation tick={event_tick} amount={estimated_predation} "
            f"(pending={pending_predation_by_tick[event_tick]})"
        )


connector.client.on_message = on_agent_message
result_start, mid_start = connector.client.subscribe(start_control_topic, qos=1)
result_events, mid_events = connector.client.subscribe(events_topic, qos=0)
print(f"Subscribe result={result_start} mid={mid_start} topic={start_control_topic}")
print(f"Subscribe result={result_events} mid={mid_events} topic={events_topic}")

print("Publish topics:")
print(f"- {tick_topic}")
print(f"- {sheep_state_topic}")
print(f"- {events_topic}")
print(f"Waiting for start command on: {start_control_topic}")
print("STARTUP_OK: agent_sheep connected and ready")

Loaded config. Primary MQTT profile: 127.0.0.1:1883
MQTT connected: True
Subscribe result=0 mid=1 topic=simulated-city/sim/control/commands
Subscribe result=0 mid=2 topic=simulated-city/sim/events
Publish topics:
- simulated-city/sim/tick
- simulated-city/sim/sheep/state
- simulated-city/sim/events
Waiting for start command on: simulated-city/sim/control/commands
STARTUP_OK: agent_sheep connected and ready


START command received on simulated-city/sim/control/commands at 2026-03-05T11:34:00.634940+00:00


In [2]:
# Cell purpose: define Phase 3 simulation parameters and deterministic seed.
simulation_cfg = config.simulation

params = SheepSimulationParams(
    grid_width=(simulation_cfg.grid_width if simulation_cfg is not None else 10),
    grid_height=(simulation_cfg.grid_height if simulation_cfg is not None else 10),
    initial_sheep=(simulation_cfg.initial_sheep if simulation_cfg is not None else 30),
    initial_sheep_energy=(simulation_cfg.sheep_initial_energy if simulation_cfg is not None else 8),
    sheep_move_cost=(simulation_cfg.sheep_move_cost if simulation_cfg is not None else 1),
    sheep_eat_gain=(simulation_cfg.sheep_eat_gain if simulation_cfg is not None else 4),
    sheep_reproduction_threshold=(
        simulation_cfg.sheep_reproduction_threshold if simulation_cfg is not None else 10
    ),
    sheep_reproduction_probability=(
        simulation_cfg.sheep_reproduction_probability if simulation_cfg is not None else 0.08
    ),
    grass_regrow_ticks=(simulation_cfg.grass_regrow_ticks if simulation_cfg is not None else 5),
)

seed = 42
if simulation_cfg is not None and simulation_cfg.seed is not None:
    seed = int(simulation_cfg.seed)

tick_count = 300
print(f"Using seed={seed}, ticks={tick_count}, grid={params.grid_width}x{params.grid_height}")

Using seed=42, ticks=300, grid=10x10


In [3]:
# Cell purpose: wait for start command, then run simulation and publish tick/state/event payloads.
start_wait_timeout_s = 15
waited_s = 0.0
print("Waiting for START command from dashboard...")
while not start_requested and waited_s < start_wait_timeout_s:
    time.sleep(0.2)
    waited_s += 0.2

if not start_requested:
    print(
        f"No START command received after {start_wait_timeout_s}s; "
        "continuing with local auto-start."
    )
    start_requested = True

print("START condition met. Running simulation...")
summaries = run_simulation(params=params, ticks=tick_count, seed=seed)
tick_interval_s = simulation_cfg.tick_interval_s if simulation_cfg is not None else 0.5
predation_wait_s = 0.05
print(f"Publishing live stream with tick interval: {tick_interval_s}s")
print(f"Predation synchronization window per tick: {predation_wait_s}s")

total_predation_applied = 0
print("tick sheep grass births deaths predation_applied ate avg_energy")
for summary in summaries:
    tick_payload = {
        "tick": summary.tick,
        "seed": seed,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    publisher.publish_json(tick_topic, json.dumps(tick_payload), qos=0)

    if predation_wait_s > 0:
        time.sleep(predation_wait_s)

    predation_applied = int(pending_predation_by_tick.pop(summary.tick, 0))
    predation_applied = min(max(0, predation_applied), int(summary.sheep_count))
    total_predation_applied += predation_applied

    adjusted_sheep_count = max(0, int(summary.sheep_count) - predation_applied)
    adjusted_deaths = int(summary.deaths) + predation_applied

    state_payload = {
        "tick": summary.tick,
        "seed": seed,
        "sheep_count": adjusted_sheep_count,
        "grass_grown_cells": summary.grass_grown_cells,
        "average_energy": round(summary.average_energy, 3),
    }
    event_payload = {
        "tick": summary.tick,
        "seed": seed,
        "source": "sheep",
        "births": summary.births,
        "deaths": adjusted_deaths,
        "sheep_natural_deaths": summary.deaths,
        "sheep_predation_deaths": predation_applied,
        "sheep_ate_grass": summary.sheep_ate_grass,
    }

    publisher.publish_json(sheep_state_topic, json.dumps(state_payload), qos=0)
    publisher.publish_json(events_topic, json.dumps(event_payload), qos=0)

    print(
        f"{summary.tick:>4} {adjusted_sheep_count:>5} {summary.grass_grown_cells:>5} "
        f"{summary.births:>6} {adjusted_deaths:>6} {predation_applied:>16} "
        f"{summary.sheep_ate_grass:>3} {summary.average_energy:>10.2f} | published"
    )

    if tick_interval_s > 0:
        time.sleep(tick_interval_s)

final_summary = summaries[-1]
final_predation_last_tick = min(
    max(0, int(pending_predation_by_tick.get(final_summary.tick, 0))),
    int(final_summary.sheep_count),
)
final_adjusted_sheep_count = max(0, int(final_summary.sheep_count) - final_predation_last_tick)
print()
print("Final summary:")
print(
    {
        "tick": final_summary.tick,
        "simulated_sheep_count": final_summary.sheep_count,
        "published_sheep_count": final_adjusted_sheep_count,
        "total_predation_applied": total_predation_applied,
        "grass_grown_cells": final_summary.grass_grown_cells,
        "average_energy": round(final_summary.average_energy, 3),
    }
)

Waiting for START command from dashboard...
START condition met. Running simulation...
Publishing live stream with tick interval: 0.5s
Predation synchronization window per tick: 0.05s
tick sheep grass births deaths predation_applied ate avg_energy
   1    30    73      0      0                0  27      10.60 | published
   2    33    50      3      0                0  23      11.52 | published
   3    36    33      3      0                0  17      11.53 | published
   4    39    27      3      0                0   6      10.33 | published
   5    42    50      3      0                0   4       9.05 | published
   6    42    54      0      0                0  19       9.86 | published
   7    43    55      1      0                0  16      10.14 | published
   8    46    48      3      0                0  13       9.67 | published
   9    48    41      2      0                0  11       9.23 | published
Queued wolf predation tick=10 amount=2 (pending=2)
  10    47    54      1   

In [4]:
# Cell purpose: verify reproducibility and disconnect MQTT cleanly.
first_run = run_simulation(params=params, ticks=tick_count, seed=seed)
second_run = run_simulation(params=params, ticks=tick_count, seed=seed)

same = [
    (
        a.tick == b.tick
        and a.sheep_count == b.sheep_count
        and a.grass_grown_cells == b.grass_grown_cells
        and a.births == b.births
        and a.deaths == b.deaths
        and a.sheep_ate_grass == b.sheep_ate_grass
        and round(a.average_energy, 8) == round(b.average_energy, 8)
    )
    for a, b in zip(first_run, second_run)
]

print(f"Deterministic run check: {all(same)}")

connector.disconnect()
print("MQTT disconnected.")

Deterministic run check: True


Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


MQTT disconnected.
